# Tangent-Space Consistency (TSC): Pilot Study

**Model:** Qwen2.5-1.5B-Instruct &nbsp;·&nbsp; **Data:** MATH500, first 250 questions &nbsp;·&nbsp; **Hardware:** Kaggle, 2 × T4

This notebook tests whether a simple geometric score on the model's hidden states can tell when the model's math answer is wrong, **without using any labels**.

**The idea.** While the model reasons, take the hidden states of the reasoning tokens and find the few main directions they lie along (with SVD). Then check how much of each *answer* token's hidden state lies **outside** those directions. If the answer does not follow from the reasoning, this "residual" may be larger.

**What the notebook does**

1. Generates 3 answers per question (1 greedy + 2 sampled), using both GPUs.
2. Grades each answer against the gold answer.
3. Computes the TSC score, three "null" versions of it, and standard baselines.
4. Compares all scores with AUROC and bootstrap confidence intervals.
5. Applies a go / no-go rule that was set before the run.

**How to run.** Settings → Accelerator = *GPU T4 x2*, Internet = *ON*. Set `CSV_PATH` in the configuration cell, then *Run All*.
Generations are cached in `OUT_DIR`, so a re-run skips generation.

**Data hygiene.** The CSV is split into `X` (questions only) and `Y` (answers only). The model only ever sees `X`. `Y` is used only after generation, for grading.

---

### Contents

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Configuration |
| 2 | Data |
| 3 | Model |
| 4 | Answer extraction and grading tools |
| 5 | Generation (both GPUs) |
| 6 | Grading |
| 7 | Pass A: re-encode generations |
| 8 | Folds and robust standardisation |
| 9 | TSC core |
| 10 | Pass B: TSC scores and nulls |
| 11 | Label-free confound adjustment |
| 12 | Feature table and baselines |
| 13 | Metrics and main comparison |
| 14 | Hypothesis checks |
| 15 | Go / no-go decision |
| 16 | Figure and saved results |
| 17 | Results of the pilot run |

## 0. Setup

Installs [math-verify](https://github.com/huggingface/Math-Verify), which checks whether two math answers are equal (for example `\frac{1}{2}` and `0.5`).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "math-verify[antlr4_13_2]"], check=False)

## 1. Configuration

All settings are in this one cell.

| Group | Setting | Value |
|---|---|---|
| Data | `CSV_PATH` | CSV with `question` and `answer` columns |
| Generation | answers per question | 1 greedy + `N_SAMPLES` = 2 sampled |
| Generation | sampling | temperature 0.7, top-p 0.95, max 1024 new tokens |
| TSC | `DEPTHS` | relative layer depths 0.4 / 0.6 / 0.8 |
| TSC | `ANCHORS` | which reasoning tokens build the subspace: all, or the last 64 / 128 / 256 |
| TSC | `TAU` | keep enough directions to explain 90% of the variance |
| TSC | `CLIP_C` | clip standardised states to ±10 |
| TSC | `LOCAL_K` | window size for TSC-Local |

**The primary score is fixed here, before any labels are seen:** depth 0.6 (layer 17) with the last 128 reasoning tokens as the anchor.

In [ ]:
import os, re, json, glob, time, random, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

CSV_PATH       = "math500.csv"                         # CSV with question / answer columns
N_QUESTIONS    = 250                                   # first 250 rows
MODEL_NAME     = "Qwen/Qwen2.5-1.5B-Instruct"          # or "Qwen/Qwen2.5-Math-1.5B-Instruct"
OUT_DIR        = "outputs"

SYSTEM_PROMPT  = "Please reason step by step, and put your final answer within \\boxed{}."
N_SAMPLES      = 2          # re-samples per question (proposal: 2)
TEMPERATURE    = 0.7
TOP_P          = 0.95
MAX_NEW_TOKENS = 1024
GEN_BATCH      = 64         # prompts per generate() call PER GPU (auto-halves on OOM)
N_GPUS_MAX     = 2          # use up to this many GPUs (Kaggle T4 x2 -> 2)
FORCE_FP32     = False      # set True only if the run reports non-finite hidden states

DEPTHS         = [0.4, 0.6, 0.8]                       # relative layer depths
ANCHORS        = ["all", "last64", "last128", "last256"]
TAU            = 0.90       # explained-variance threshold for the rank m
CLIP_C         = 10.0       # clip after robust standardisation
LOCAL_K        = 20         # TSC-Local window
MIN_REASONING  = 16         # need at least this many reasoning tokens before the answer
CALIB_PER_GEN  = 32         # token states per generation used for median/MAD

# Pre-registered primary score — fixed BEFORE looking at labels. Do not change after seeing results.
PRIMARY_DEPTH  = 0.6
PRIMARY_ANCHOR = "last128"

N_BOOT         = 2000       # cluster-bootstrap resamples (over questions)
SEED           = 0

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DT = {d: f"d{int(round(d * 100))}" for d in DEPTHS}   # column tag per depth, e.g. 0.6 -> "d60"

## 2. Data

Loads the first `N_QUESTIONS` rows and splits them into:

- **`X`**: the questions. This is the only thing the model sees.
- **`Y`**: the gold answers. These are used only for grading, after generation.

In [ ]:
def locate_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"CSV not found at {path}. Set CSV_PATH in the configuration cell.")
    return path

raw = pd.read_csv(locate_csv(CSV_PATH), dtype=str, keep_default_na=False)
colmap = {c.strip().lower(): c for c in raw.columns}
assert "question" in colmap and "answer" in colmap, f"Expected question/answer columns, found {list(raw.columns)}"
raw = raw.iloc[:N_QUESTIONS].reset_index(drop=True)

X = pd.DataFrame({"qid": np.arange(len(raw)), "question": raw[colmap["question"]].astype(str)})
Y = pd.DataFrame({"qid": np.arange(len(raw)), "answer": raw[colmap["answer"]].astype(str).str.strip().str.strip("$").str.strip()})
del raw   # from here on, the model side only ever reads X

print(f"X: {X.shape}  (fed to the model)")
print(f"Y: {Y.shape}  (used only for grading, after generation)")
print(X.head(3).to_string(index=False, max_colwidth=80))

## 3. Model

Loads the tokenizer and the model.

- Uses float16 on GPU. A quick check on a few prompts switches to float32 if float16 produces `inf`/`nan` hidden states.
- Sets a clean generation config, so greedy decoding is truly greedy (no default repetition penalty or top-k).
- Maps each relative depth to a layer index. For the 28-layer model: 0.4 → 11, 0.6 → 17, 0.8 → 22.

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEV.type != "cuda":
    print("WARNING: no GPU found. Set Accelerator = GPU T4 x2 in the notebook settings.")
DTYPE = torch.float32 if (FORCE_FP32 or DEV.type == "cpu" or "math" in MODEL_NAME.lower()) else torch.float16

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
_tv = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
_dkw = {"dtype": DTYPE} if _tv >= (4, 56) else {"torch_dtype": DTYPE}
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_dkw)
model = model.to(device=DEV, dtype=DTYPE).eval()     # explicit cast: dtype is guaranteed whatever the version

_eos = model.generation_config.eos_token_id
_eos = [_eos] if isinstance(_eos, int) else list(_eos or [])
for name in ("<|im_end|>", "<|endoftext|>"):
    tid = tok.convert_tokens_to_ids(name)
    if isinstance(tid, int) and tid != tok.unk_token_id:
        _eos.append(tid)
if tok.eos_token_id is not None:
    _eos.append(tok.eos_token_id)
EOS_IDS = sorted(set(_eos))
EOS_SET = set(EOS_IDS)
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else EOS_IDS[0]
# Clean generation config: drop the model's default repetition penalty / top-k so greedy is truly greedy
model.generation_config = GenerationConfig(eos_token_id=EOS_IDS, pad_token_id=PAD_ID,
                                           bos_token_id=model.generation_config.bos_token_id)

N_LAYERS = model.config.num_hidden_layers
D_MODEL = model.config.hidden_size
LAYER_OF = {d: int(round(d * N_LAYERS)) for d in DEPTHS}       # index into hidden_states (0 = embeddings)
SPECIAL_IDS = set(tok.all_special_ids)
DEC = dict(skip_special_tokens=False, clean_up_tokenization_spaces=False)
print(f"{MODEL_NAME} | {DTYPE} | {DEV} | layers={N_LAYERS} D={D_MODEL} | depth->layer {LAYER_OF} | eos={EOS_IDS}")

def build_prompt(question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

PROMPT_IDS = [tok(build_prompt(q), add_special_tokens=False).input_ids for q in X["question"]]

@torch.no_grad()
def numerics_ok(n_check=8):
    for q in range(min(n_check, len(PROMPT_IDS))):
        out = model(input_ids=torch.tensor([PROMPT_IDS[q]], device=DEV), output_hidden_states=True, use_cache=False)
        if not torch.isfinite(out.logits).all() or not all(torch.isfinite(h).all() for h in out.hidden_states):
            return False
    return True

if DTYPE == torch.float16 and not numerics_ok():
    print("fp16 overflow detected -> switching the model to float32")
    DTYPE = torch.float32
    model = model.float()
print("numerics check passed:", DTYPE)

DEVICES = [torch.device(f"cuda:{i}") for i in range(min(torch.cuda.device_count(), N_GPUS_MAX))] or [DEV]
print("devices used:", [f"{d} ({torch.cuda.get_device_name(d)})" for d in DEVICES] if DEV.type == "cuda" else DEVICES)
if DEV.type == "cuda" and len(DEVICES) < 2:
    print("NOTE: only one GPU visible — pick 'GPU T4 x2' in Settings to use both.")

## 4. Answer extraction and grading tools

- `last_boxed_span`: finds the content of the last `\boxed{...}` and its character position.
- `equiv`: checks whether a predicted answer equals the gold answer (normalised string match first, then math-verify).
- `char_to_token`: maps a character position back to a token index, so we know exactly which tokens form the answer.
- `token_class`: a coarse token type (digit / operator / word / punctuation / special), used later in the confound adjustment.

In [ ]:
def last_boxed_span(text):
    """Content of the last \\boxed{...}, with char offsets [start, end) of the content."""
    idx = text.rfind("\\boxed")
    if idx < 0:
        return None
    i = idx + len("\\boxed")
    while i < len(text) and text[i] == " ":
        i += 1
    if i >= len(text) or text[i] != "{":
        return None
    depth = 0
    for j in range(i, len(text)):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                content = text[i + 1:j]
                if not content.strip():
                    return None
                lead = len(content) - len(content.lstrip())
                trail = len(content) - len(content.rstrip())
                return content.strip(), i + 1 + lead, j - trail
    return None   # unbalanced (e.g. truncated)

def _norm(s):
    s = str(s).strip().replace("\n", "")
    s = re.sub(r"\\(left|right)", "", s)
    s = s.replace("\\!", "").replace("\\,", "").replace("\\;", "").replace("\\ ", "")
    s = s.replace("dfrac", "frac").replace("tfrac", "frac")
    s = s.replace("^\\circ", "").replace("^{\\circ}", "").replace("\\circ", "")
    s = s.replace("\\$", "").replace("$", "").replace("\\%", "").replace("%", "")
    s = re.sub(r"\\(text|mbox|mathrm)\{\s*([^{}]*)\}", r"\2", s)
    s = re.sub(r"^[a-zA-Z]\s*(=|\\in)\s*", "", s)
    s = s.replace(" ", "").rstrip(".")
    if re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", s):
        s = s.replace(",", "")
    if s.startswith("."):
        s = "0" + s
    return s

try:
    from math_verify import parse as mv_parse, verify as mv_verify
    HAVE_MV = True
except Exception as e:
    HAVE_MV = False
    print("math-verify unavailable; falling back to normalised string match:", e)

_PARSE = {}
def _mv(s):
    if s not in _PARSE:
        try:
            _PARSE[s] = mv_parse("$" + s + "$")
        except Exception:
            _PARSE[s] = []
    return _PARSE[s]

def equiv(gold, pred):
    if gold is None or pred is None:
        return False
    if _norm(gold) == _norm(pred):
        return True
    if HAVE_MV:
        g, p = _mv(gold), _mv(pred)
        if g and p:
            try:
                return bool(mv_verify(g, p))
            except Exception:
                return False
    return False

def char_to_token(ids, char_pos):
    """Index of the generated token that covers character `char_pos` of decode(ids)."""
    cache = {}
    def plen(k):
        if k not in cache:
            cache[k] = len(tok.decode(ids[:k], **DEC))
        return cache[k]
    lo, hi = 0, len(ids) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if plen(mid + 1) > char_pos:
            hi = mid
        else:
            lo = mid + 1
    return lo

_CLS = {}
CLS_NAMES = ["digit", "operator", "word", "punct", "special"]
def token_class(tid):
    if tid not in _CLS:
        if tid in SPECIAL_IDS:
            c = 4
        else:
            s = tok.decode([tid], **DEC).strip()
            if s == "":
                c = 3
            elif re.fullmatch(r"[0-9.]+", s):
                c = 0
            elif re.search(r"[A-Za-z]", s) and not s.startswith("\\"):
                c = 2
            elif re.fullmatch(r"[.,;:!?'\"()\[\]{}]+", s):
                c = 3
            else:
                c = 1
        _CLS[tid] = c
    return _CLS[tid]

## 5. Generation (both GPUs)

Runs one independent worker process per GPU. Each worker writes its answers to a `.jsonl` file as it goes.

- 1 greedy answer and `N_SAMPLES` sampled answers per question.
- Prompts are sorted by length, so each batch has similar lengths.
- If a batch runs out of GPU memory, it is split in half automatically.
- Finished answers are cached, so an interrupted run only generates what is missing.

In [ ]:
import subprocess, threading
from concurrent.futures import ThreadPoolExecutor, wait

WORKER_SRC = r'''
import json, re, sys
import torch, transformers
from transformers import AutoModelForCausalLM, GenerationConfig

job = json.load(open(sys.argv[1]))
dev = torch.device("cuda:0")                 # this process sees exactly one GPU (CUDA_VISIBLE_DEVICES)
dtype = getattr(torch, job["dtype"])
tv = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
kw = {"dtype": dtype} if tv >= (4, 56) else {"torch_dtype": dtype}
model = AutoModelForCausalLM.from_pretrained(job["model"], **kw).to(device=dev, dtype=dtype).eval()
model.generation_config = GenerationConfig(eos_token_id=job["eos_ids"], pad_token_id=job["pad_id"],
                                           bos_token_id=model.generation_config.bos_token_id)
EOS, PAD, N, B = set(job["eos_ids"]), job["pad_id"], job["n_samples"], job["batch"]
PROMPTS = {int(k): v for k, v in job["prompt_ids"].items()}

def cut(ids):
    for i, t in enumerate(ids):
        if t in EOS:
            return ids[:i], False
    return ids, True                         # no EOS -> truncated

@torch.no_grad()
def gen(qids, sample):
    seqs = [PROMPTS[q] for q in qids]
    T = max(map(len, seqs))
    inp = torch.full((len(seqs), T), PAD, dtype=torch.long)
    att = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):             # left padding
        inp[i, T - len(s):] = torch.tensor(s)
        att[i, T - len(s):] = 1
    kw = dict(max_new_tokens=job["max_new_tokens"], do_sample=sample, pad_token_id=PAD, eos_token_id=job["eos_ids"])
    if sample:
        kw.update(temperature=job["temperature"], top_p=job["top_p"], top_k=0, num_return_sequences=N)
    out = model.generate(input_ids=inp.to(dev), attention_mask=att.to(dev), **kw)
    if not torch.is_tensor(out):
        out = out.sequences
    return out[:, T:].cpu().tolist()

def gen_safe(qids, sample):
    try:
        return gen(qids, sample)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(qids) == 1:
            raise
        h = len(qids) // 2
        print(f"OOM at batch {len(qids)} -> splitting", flush=True)
        return gen_safe(qids[:h], sample) + gen_safe(qids[h:], sample)

with open(job["out_path"], "a") as f:
    def emit(q, s, ids):
        g, trunc = cut(ids)
        f.write(json.dumps(dict(qid=int(q), sample_idx=int(s), kind="greedy" if s == 0 else "sample",
                                gen_ids=g, truncated=bool(trunc))) + "\n")
    greedy = job["greedy"]
    for b in range(0, len(greedy), B):
        qs = greedy[b:b + B]
        for q, ids in zip(qs, gen_safe(qs, False)):
            emit(q, 0, ids)
        f.flush()
        print(f"greedy {min(b + B, len(greedy))}/{len(greedy)}", flush=True)
    sample = job["sample"]                   # [[qid, [missing sample indices]], ...]
    sb = max(1, B // N)
    for b in range(0, len(sample), sb):
        chunk = sample[b:b + sb]
        qs = [q for q, _ in chunk]
        torch.manual_seed(job["seed"] * 100003 + qs[0])
        outs = gen_safe(qs, True)
        for i, (q, missing) in enumerate(chunk):
            for s in range(N):
                if s + 1 in missing:
                    emit(q, s + 1, outs[i * N + s])
        f.flush()
        print(f"samples {min(b + sb, len(sample))}/{len(sample)}", flush=True)
print("WORKER_DONE", flush=True)
'''

def read_cached():
    recs = []
    for path in sorted(glob.glob(os.path.join(OUT_DIR, "generations*.jsonl"))):
        with open(path) as fh:
            for line in fh:
                try:
                    recs.append(json.loads(line))
                except json.JSONDecodeError:
                    pass                              # half-written last line from an interrupted run
    return [r for r in recs if r["qid"] < len(X)]

def count_lines():
    n = 0
    for path in glob.glob(os.path.join(OUT_DIR, "generations*.jsonl")):
        with open(path) as fh:
            n += sum(1 for _ in fh)
    return n

def physical_gpu(dv):
    vis = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
    return vis.split(",")[dv.index].strip() if vis else str(dv.index)

records = read_cached()
if records:
    print(f"loaded {len(records)} cached generations")
done = {(r["qid"], r["sample_idx"]) for r in records}
order = sorted(range(len(X)), key=lambda q: len(PROMPT_IDS[q]))       # similar lengths per batch
greedy_todo = [q for q in order if (q, 0) not in done]
sample_todo = [[q, [s for s in range(1, N_SAMPLES + 1) if (q, s) not in done]] for q in order]
sample_todo = [x for x in sample_todo if x[1]]
total_todo = len(greedy_todo) + sum(len(m) for _, m in sample_todo)

if total_todo:
    if DEV.type != "cuda":
        raise RuntimeError("Generation needs a GPU: set Accelerator = GPU T4 x2.")
    worker_path = os.path.join(OUT_DIR, "gen_worker.py")
    with open(worker_path, "w") as fh:
        fh.write(WORKER_SRC)
    nW = len(DEVICES)
    procs, t0 = [], time.time()
    model.to("cpu")                                   # free GPU 0 for its worker during generation
    torch.cuda.empty_cache()
    try:
        for k, dv in enumerate(DEVICES):
            job = dict(model=MODEL_NAME, dtype=str(DTYPE).replace("torch.", ""), eos_ids=EOS_IDS, pad_id=PAD_ID,
                       max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P,
                       n_samples=N_SAMPLES, batch=GEN_BATCH, seed=SEED,
                       greedy=greedy_todo[k::nW], sample=sample_todo[k::nW],
                       out_path=os.path.join(OUT_DIR, f"generations_gpu{dv.index}.jsonl"))
            need = set(job["greedy"]) | {q for q, _ in job["sample"]}
            job["prompt_ids"] = {str(q): PROMPT_IDS[q] for q in need}
            job_path = os.path.join(OUT_DIR, f"gen_job_gpu{dv.index}.json")
            with open(job_path, "w") as fh:
                json.dump(job, fh)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=physical_gpu(dv), PYTHONUNBUFFERED="1",
                       TOKENIZERS_PARALLELISM="false")
            log = open(os.path.join(OUT_DIR, f"gen_worker_gpu{dv.index}.log"), "w")
            procs.append((dv, subprocess.Popen([sys.executable, worker_path, job_path], env=env,
                                               stdout=log, stderr=subprocess.STDOUT), log))
        print(f"launched {nW} generation worker(s): " + ", ".join(f"GPU {physical_gpu(dv)}" for dv, _, _ in procs))
        base = count_lines()
        bar = tqdm(total=total_todo, desc=f"generation on {nW} GPU(s)")
        try:
            while any(p.poll() is None for _, p, _ in procs):
                time.sleep(10)
                bar.n = min(total_todo, count_lines() - base)
                bar.refresh()
        except KeyboardInterrupt:
            for _, p, _ in procs:
                p.terminate()
            raise
        bar.n = min(total_todo, count_lines() - base)
        bar.refresh()
        bar.close()
    finally:
        for _, p, log in procs:
            if p.poll() is None:
                p.terminate()
            log.close()
        model.to(DEV)                                 # main-process model back on GPU 0
    failed = [(dv, p.returncode) for dv, p, _ in procs if p.returncode != 0]
    for dv, rc in failed:
        print(f"--- worker on GPU {physical_gpu(dv)} exited with code {rc}; last lines of its log:")
        with open(os.path.join(OUT_DIR, f"gen_worker_gpu{dv.index}.log")) as fh:
            print("".join(fh.readlines()[-40:]))
    if failed:
        raise RuntimeError("A generation worker failed. Re-run this cell: finished generations are kept and only the rest is generated.")
    print(f"generation done in {(time.time() - t0) / 60:.1f} min")

records = sorted({(r["qid"], r["sample_idx"]): r for r in read_cached()}.values(),
                 key=lambda r: (r["qid"], r["sample_idx"]))
expected = len(X) * (1 + N_SAMPLES)
if len(records) < expected:
    raise RuntimeError(f"only {len(records)}/{expected} generations on disk; re-run this cell to finish them")
for gid, r in enumerate(records):
    r["gid"] = gid
print(f"{len(records)} generations ready. Example greedy output (end):\n",
      tok.decode(records[0]["gen_ids"], **DEC)[-400:])

## 6. Grading

`Y` is used here for the first time.

For each answer, this cell:

- extracts the last `\boxed{}` answer and finds its tokens;
- marks it **valid** if it has a boxed answer and at least `MIN_REASONING` = 16 reasoning tokens before it;
- marks it **correct** if it matches the gold answer.

It also records:

- **agreement**: how many of the other answers to the same question give the same result;
- **changed**: whether a sampled answer differs from the greedy one.

In [ ]:
gold = dict(zip(Y["qid"], Y["answer"]))
for r in records:
    ids = r["gen_ids"]
    text = tok.decode(ids, **DEC)
    span = last_boxed_span(text)
    r["pred"], r["a"], r["L"] = None, -1, 0
    if span is not None and len(ids) > 0:
        pred, s, e = span
        a = char_to_token(ids, s)
        b = char_to_token(ids, e - 1)
        r["pred"], r["a"], r["L"] = pred, a, b - a + 1
    r["n_gen"] = len(ids)
    r["format_ok"] = r["pred"] is not None
    r["valid"] = bool(r["format_ok"] and r["a"] >= MIN_REASONING and r["L"] >= 1 and r["a"] + r["L"] <= len(ids))
    r["correct"] = bool(r["format_ok"] and equiv(gold[r["qid"]], r["pred"]))

by_q = defaultdict(list)
for r in records:
    by_q[r["qid"]].append(r)
for q, rs in by_q.items():
    greedy = next((x for x in rs if x["sample_idx"] == 0), None)
    for r in rs:
        others = [x for x in rs if x is not r]
        r["agreement"] = float(np.mean([equiv(r["pred"], o["pred"]) for o in others])) if (others and r["format_ok"]) else 0.0
        if r["sample_idx"] > 0 and greedy is not None and greedy["format_ok"] and r["format_ok"]:
            r["changed"] = not equiv(greedy["pred"], r["pred"])
        else:
            r["changed"] = None

summ = pd.DataFrame([{k: r[k] for k in ("kind", "format_ok", "valid", "correct", "truncated", "n_gen")} for r in records])
print(summ.groupby("kind").agg(n=("valid", "size"), format_ok=("format_ok", "mean"), valid=("valid", "mean"),
                               accuracy=("correct", "mean"), truncated=("truncated", "mean"),
                               mean_len=("n_gen", "mean")).round(3).to_string())
vr = [r for r in records if r["valid"]]
print(f"analysis set: {len(vr)} valid generations, {sum(not r['correct'] for r in vr)} wrong / {sum(r['correct'] for r in vr)} correct")

## 7. Pass A: re-encode generations

Each answer is run through the model once more (one model copy per GPU). For every generated token this collects:

- **surprisal** (negative log-prob) and **entropy**, used for the baselines and the confound adjustment;
- a random set of 32 hidden states per answer, used to compute the standardisation statistics;
- the hidden states of the answer tokens.

In [ ]:
# One model replica per GPU; each GPU processes its own share of generations in a thread.
_dkey = "dtype" if _tv >= (4, 56) else "torch_dtype"
REPLICAS = {str(DEVICES[0]): model}
for dv in DEVICES[1:]:
    REPLICAS[str(dv)] = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **{_dkey: DTYPE}).to(device=dv, dtype=DTYPE).eval()
print("model replicas on:", list(REPLICAS))

def split_balanced(items, cost):
    """Longest-first assignment so every GPU gets about the same number of tokens."""
    loads, shards = [0] * len(DEVICES), [[] for _ in DEVICES]
    for it in sorted(items, key=cost, reverse=True):
        k = int(np.argmin(loads))
        shards[k].append(it)
        loads[k] += cost(it)
    return list(zip(DEVICES, shards))

def run_on_gpus(fn, shards, desc):
    """Run fn(device, shard, tick) in one thread per GPU; progress bar is drawn from the main thread."""
    total = sum(len(s) for _, s in shards)
    counter, lock = [0], threading.Lock()
    def tick():
        with lock:
            counter[0] += 1
    bar = tqdm(total=total, desc=f"{desc} on {len(shards)} GPU(s)")
    with ThreadPoolExecutor(max_workers=len(shards)) as ex:
        futs = [ex.submit(fn, dv, shard, tick) for dv, shard in shards]
        while True:
            finished, _ = wait(futs, timeout=2)
            bar.n = counter[0]
            bar.refresh()
            if len(finished) == len(futs):
                break
    bar.close()
    return [f.result() for f in futs]      # re-raises any worker exception here

def surprisal_entropy(logits, targets, chunk=256):
    s_all, e_all = [], []
    for i in range(0, len(targets), chunk):
        lsm = torch.log_softmax(logits[i:i + chunk].float(), -1)
        s_all.append(-lsm.gather(1, targets[i:i + chunk, None])[:, 0])
        e_all.append(-(lsm.exp() * lsm).sum(-1))
    return torch.cat(s_all), torch.cat(e_all)

# token classes are computed here in the main thread (the tokenizer is not thread-safe)
CLS_OF = {r["gid"]: np.array([token_class(t) for t in r["gen_ids"]], dtype=np.int64) for r in records}
TOKF = {}
CALIB = {d: {} for d in DEPTHS}
ANS_RAW = {d: {} for d in DEPTHS}

def pass_a(dv, shard, tick):
    if dv.type == "cuda":
        torch.cuda.set_device(dv)
    mdl, bad = REPLICAS[str(dv)], 0
    for r in shard:
        gid, ids, n = r["gid"], r["gen_ids"], len(r["gen_ids"])
        if n == 0:
            tick()
            continue
        p = PROMPT_IDS[r["qid"]]
        P = len(p)
        inp = torch.tensor([p + ids], device=dv)
        with torch.no_grad():
            out = mdl(input_ids=inp, output_hidden_states=True, use_cache=False)
        surp, ent = surprisal_entropy(out.logits[0, P - 1:P + n - 1], inp[0, P:P + n])
        TOKF[gid] = dict(surp=surp.cpu().numpy(), ent=ent.cpu().numpy(), cls=CLS_OF[gid])
        rng = np.random.default_rng(SEED * 1_000_003 + gid)          # per-generation RNG: thread-safe, reproducible
        sel = torch.as_tensor(np.sort(rng.choice(n, size=min(n, CALIB_PER_GEN), replace=False)), device=dv)
        for d in DEPTHS:
            H = out.hidden_states[LAYER_OF[d]][0, P:P + n].float()
            if not torch.isfinite(H).all():
                bad += 1
            CALIB[d][gid] = H[sel].cpu().numpy()
            if r["valid"]:
                ANS_RAW[d][gid] = H[r["a"]:r["a"] + r["L"]].cpu().numpy()
        del out
        tick()
    return bad

t0 = time.time()
n_nonfinite = sum(run_on_gpus(pass_a, split_balanced(records, lambda r: len(PROMPT_IDS[r["qid"]]) + len(r["gen_ids"])),
                              "pass A (re-encode)"))
print(f"pass A done in {(time.time() - t0) / 60:.1f} min")
if n_nonfinite:
    raise RuntimeError(f"{n_nonfinite} generations have non-finite hidden states. Set FORCE_FP32 = True in cell 1 "
                       "and Run All again (cached generations are reused, only re-encoding repeats).")

## 8. Folds and robust standardisation

- Questions are split into **2 folds**. All answers to one question stay in the same fold.
- Each hidden dimension is centred with its median and scaled with its MAD, then clipped to ±`CLIP_C`. The statistics for one fold come from the **other** fold, so no answer is used to scale itself.
- For Null 3, each answer gets a partner answer from a **different question** in the same fold.

In [ ]:
qfold = np.random.default_rng(SEED + 7).permutation(len(X)) % 2      # folds split by question
for r in records:
    r["fold"] = int(qfold[r["qid"]])

def robust_stats(arr):
    med = np.median(arr, 0)
    mad = 1.4826 * np.median(np.abs(arr - med), 0)
    return med, np.maximum(mad, max(1e-6, 1e-3 * float(np.median(mad))))

STATS = {}
for f in (0, 1):
    for d in DEPTHS:   # statistics for fold f come from the OTHER fold's unlabelled generations
        src = [CALIB[d][r["gid"]] for r in records if r["fold"] != f and r["gid"] in CALIB[d]]
        med, sc = robust_stats(np.concatenate(src))
        for dv in DEVICES:                                            # one copy per GPU
            STATS[(str(dv), f, d)] = (torch.tensor(med, device=dv), torch.tensor(sc, device=dv))
del CALIB

def standardise(H, f, d):
    med, sc = STATS[(str(H.device), f, d)]
    return ((H - med) / sc).clamp(-CLIP_C, CLIP_C)

rngP = np.random.default_rng(SEED + 11)
valid_ids = [r["gid"] for r in records if r["valid"]]
PARTNER, INV_PARTNER = {}, defaultdict(list)
for g in valid_ids:
    cands = [h for h in valid_ids if records[h]["fold"] == records[g]["fold"] and records[h]["qid"] != records[g]["qid"]]
    if cands:
        PARTNER[g] = int(rngP.choice(cands))
        INV_PARTNER[PARTNER[g]].append(g)
INV_PARTNER = dict(INV_PARTNER)

## 9. TSC core

The geometric part of the method.

- `anchor_states`: the reasoning hidden states used to build the subspace (all of them, or only the last K).
- `subspace`: centres the anchor states and runs SVD. It keeps the smallest number of directions **m** that explain at least `TAU` = 90% of the variance.
- `residual`: for a hidden state *z*, computes the **relative residual**

$$\rho = \frac{\lVert (I - VV^\top)(z - \mu) \rVert}{\lVert z - \mu \rVert}$$

  This is the share of *z* (relative to the anchor centre μ) that lies **outside** the subspace. It ranges from 0 to 1.
- `tsc_local`: the same idea, but the subspace is built from only the `LOCAL_K` = 20 tokens just before each token. It is computed efficiently with the small k × k Gram matrix.

In [ ]:
def anchor_states(Z, end, anchor):
    if anchor == "all":
        return Z[:end]
    K = int(anchor[4:])
    return Z[max(0, end - K):end]

def _svd(Xc):
    try:
        return torch.linalg.svd(Xc, full_matrices=False)
    except RuntimeError:                                    # rare GPU non-convergence -> CPU float64
        U, S, Vh = torch.linalg.svd(Xc.detach().double().cpu(), full_matrices=False)
        return U.to(Xc), S.to(Xc), Vh.to(Xc)

def _eigh(G):
    try:
        return torch.linalg.eigh(G)
    except RuntimeError:
        ev, U = torch.linalg.eigh(G.cpu())
        return ev.to(G.device), U.to(G.device)

def subspace(R, tau=TAU):
    """Centred SVD of anchor states; rank m = smallest m explaining >= tau of variance (m <= rows-1)."""
    mu = R.mean(0)
    Xc = R - mu
    if R.shape[0] < 2:
        return mu, None
    _, S, Vh = _svd(Xc)
    s2 = S ** 2
    tot = s2.sum()
    if tot <= 0:
        return mu, None
    m = int((torch.cumsum(s2, 0) / tot < tau).sum().item()) + 1
    m = max(1, min(m, R.shape[0] - 1, int((S > 1e-6 * S[0]).sum().item())))
    return mu, Vh[:m].T.contiguous()          # D x m

def residual(Zs, mu, V):
    """Relative residual rho = ||(I - VV^T)(z - mu)|| / ||z - mu|| and log ||z - mu||."""
    dv = Zs - mu
    dn2 = (dv * dv).sum(-1).clamp_min(1e-12)
    if V is None:
        rho = torch.ones_like(dn2)
    else:
        p2 = ((dv @ V) ** 2).sum(-1)
        rho = torch.sqrt((1 - p2 / dn2).clamp(0, 1))
    return rho, 0.5 * torch.log(dn2)

def tsc_local(Z, k=LOCAL_K, tau=TAU, chunk=256):
    """Per-token TSC-Local with a k-token centred window, via the k x k Gram matrix."""
    n = Z.shape[0]
    rho = torch.full((n,), float("nan"), device=Z.device)
    logn = rho.clone()
    if n <= k:
        return rho.cpu().numpy(), logn.cpu().numpy()
    win = Z.unfold(0, k, 1)                                 # win[s] = Z[s:s+k].T, shape (D, k)
    ar = torch.arange(k, device=Z.device)
    for s0 in range(k, n, chunk):
        t = torch.arange(s0, min(n, s0 + chunk), device=Z.device)
        W = win[t - k].transpose(1, 2)                      # (B, k, D) = Z[t-k:t]
        mu = W.mean(1)
        Xc = W - mu[:, None]
        G = (Xc @ Xc.transpose(1, 2)).double()
        ev, U = _eigh(G)
        ev, U = ev.flip(-1).clamp_min(0), U.flip(-1)
        cum = ev.cumsum(-1) / ev.sum(-1, keepdim=True).clamp_min(1e-12)
        m = ((cum < tau).sum(-1) + 1).clamp(max=k - 1)
        dv = Z[t] - mu
        gx = (Xc @ dv[..., None]).double()                  # X d
        c = (U.transpose(1, 2) @ gx)[..., 0] / ev.sqrt().clamp_min(1e-12)   # V^T d = S^-1 U^T X d
        p2 = torch.where(ar[None] < m[:, None], c ** 2, torch.zeros_like(c)).sum(-1)
        dn2 = (dv.double() ** 2).sum(-1).clamp_min(1e-12)
        rho[t] = torch.sqrt((1 - p2 / dn2).clamp(0, 1)).float()
        logn[t] = (0.5 * torch.log(dn2)).float()
    return rho.cpu().numpy(), logn.cpu().numpy()

## 10. Pass B: TSC scores and nulls

For every valid answer, every depth and every anchor, this cell computes:

| Score | Subspace built from | Applied to |
|---|---|---|
| **TSC-Reasoning** | this answer's reasoning tokens | this answer's answer tokens |
| **Null 1: shuffled anchor** | the same number of tokens, picked at random from all earlier tokens | this answer's answer tokens |
| **Null 2: random subspace** | random directions of the same size | this answer's answer tokens |
| **Null 3: other-question anchor** | this answer's reasoning tokens | the answer tokens of a **different** question |
| **Pre-answer token** | the reasoning tokens | the single token just before the answer |
| **TSC-Local** | the 20 previous tokens | every token |

If TSC really captures the reasoning geometry, it should beat all three nulls.

In [ ]:
# All result dicts are created up front; each thread writes only its own keys.
KINDS = ("real", "random", "shuffled", "other")
TSC = {(d, an, k): {} for d in DEPTHS for an in ANCHORS for k in KINDS}   # per-answer-token rho
LOGN = {(d, an): {} for d in DEPTHS for an in ANCHORS}                    # per-answer-token log||z - mu||
RANK = {(d, an): {} for d in DEPTHS for an in ANCHORS}                    # rank m of the anchor subspace
PRE = {d: {} for d in DEPTHS}                                             # rho at the token before the answer
LOCAL = {d: {} for d in DEPTHS}                                           # TSC-Local per token

def pass_b(dv, shard, tick):
    if dv.type == "cuda":
        torch.cuda.set_device(dv)
    body = REPLICAS[str(dv)].model
    for r in shard:
        gid, ids, a, L, f = r["gid"], r["gen_ids"], r["a"], r["L"], r["fold"]
        p = PROMPT_IDS[r["qid"]]
        P, n = len(p), len(ids)
        with torch.no_grad():
            hs = body(input_ids=torch.tensor([p + ids], device=dv), output_hidden_states=True, use_cache=False).hidden_states
        gen_t = torch.Generator(device=dv).manual_seed(SEED * 100003 + gid)
        for d in DEPTHS:
            Z = standardise(hs[LAYER_OF[d]][0, P:P + n].float(), f, d)
            ans = Z[a:a + L]
            for an in ANCHORS:
                mu, V = subspace(anchor_states(Z, a, an))
                rho, logn = residual(ans, mu, V)
                m = 0 if V is None else V.shape[1]
                TSC[(d, an, "real")][gid] = rho.cpu().numpy()
                LOGN[(d, an)][gid] = logn.cpu().numpy()
                RANK[(d, an)][gid] = m
                # Null 2: random orthonormal subspace of the same rank, same centre
                Q = torch.linalg.qr(torch.randn(Z.shape[1], max(m, 1), device=dv, generator=gen_t))[0]
                TSC[(d, an, "random")][gid] = residual(ans, mu, Q)[0].cpu().numpy()
                # Null 1: shuffled anchor — same number of states drawn at random from all earlier tokens
                if an != "all":
                    K = int(an[4:])
                    if a > K:
                        idx = torch.randperm(a, device=dv, generator=gen_t)[:K]
                        mu2, V2 = subspace(Z[idx])
                        TSC[(d, an, "shuffled")][gid] = residual(ans, mu2, V2)[0].cpu().numpy()
                    else:
                        TSC[(d, an, "shuffled")][gid] = rho.cpu().numpy()   # anchor already = all earlier tokens
                # Null 3: this anchor scores the answer of a different question (same fold, same model)
                for g2 in INV_PARTNER.get(gid, []):
                    ans2 = standardise(torch.tensor(ANS_RAW[d][g2], device=dv), records[g2]["fold"], d)
                    TSC[(d, an, "other")][g2] = residual(ans2, mu, V)[0].cpu().numpy()
            mu, V = subspace(anchor_states(Z, a - 1, PRIMARY_ANCHOR))
            PRE[d][gid] = float(residual(Z[a - 1:a], mu, V)[0].item())
            LOCAL[d][gid] = tsc_local(Z)
        del hs
        tick()

t0 = time.time()
run_on_gpus(pass_b, split_balanced([r for r in records if r["valid"]], lambda r: r["a"] + len(r["gen_ids"])), "pass B (TSC)")
print(f"pass B done in {(time.time() - t0) / 60:.1f} min")

## 11. Label-free confound adjustment

The residual can depend on simple things, such as the token type or the position in the text. To remove that effect:

1. A small gradient-boosting model predicts the residual from token surprisal, entropy, relative position, distance from the anchor centre, and token type.
2. The **adjusted score** is the real residual minus this prediction.

This step uses **no correctness labels**. It is cross-fitted: the model trained on one fold scores the other fold.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

def design(gid, t_idx, logn):
    tf = TOKF[gid]
    n = len(tf["surp"])
    oh = np.eye(5, dtype=np.float32)[tf["cls"][t_idx]]
    return np.column_stack([tf["surp"][t_idx], tf["ent"][t_idx], t_idx / n, logn, oh]).astype(np.float32)

def crossfit(gids, train_block, test_block, max_rows=200_000):
    """train_block/test_block(gid) -> (features, rho). Returns {gid: rho - g_hat} for every gid."""
    out = {}
    rng = np.random.default_rng(SEED + 3)
    for f in (0, 1):
        tr = [g for g in gids if records[g]["fold"] != f]
        te = [g for g in gids if records[g]["fold"] == f]
        if not tr or not te:
            continue
        blocks = [train_block(g) for g in tr]
        Xtr = np.concatenate([b[0] for b in blocks])
        ytr = np.concatenate([b[1] for b in blocks])
        ok = np.isfinite(ytr) & np.isfinite(Xtr).all(1)
        Xtr, ytr = Xtr[ok], ytr[ok]
        if len(ytr) > max_rows:
            keep = rng.choice(len(ytr), max_rows, replace=False)
            Xtr, ytr = Xtr[keep], ytr[keep]
        g_hat = HistGradientBoostingRegressor(max_depth=3, max_iter=300, learning_rate=0.05,
                                              min_samples_leaf=40, random_state=SEED).fit(Xtr, ytr)
        for g in te:
            Xg, yg = test_block(g)
            out[g] = yg - g_hat.predict(Xg)
    return out

def ans_idx(g):
    return np.arange(records[g]["a"], records[g]["a"] + records[g]["L"])

ADJ_R, ADJ_L = {}, {}
for d in DEPTHS:
    for an in ANCHORS:
        blk = lambda g, d=d, an=an: (design(g, ans_idx(g), LOGN[(d, an)][g]), TSC[(d, an, "real")][g])
        ADJ_R[(d, an)] = crossfit(valid_ids, blk, blk)
    def tr_local(g, d=d):       # all tokens with a full window
        rho, logn = LOCAL[d][g]
        t = np.arange(LOCAL_K, len(rho))
        return design(g, t, logn[t]), rho[t]
    def te_local(g, d=d):       # answer tokens only
        rho, logn = LOCAL[d][g]
        t = ans_idx(g)
        return design(g, t, logn[t]), rho[t]
    ADJ_L[d] = crossfit(valid_ids, tr_local, te_local)

## 12. Feature table and baselines

Builds one row per valid answer with every score. For every score, a **higher value means "more likely wrong"**.

| Baseline | Meaning |
|---|---|
| Mean token log-prob (neg) | how unsure the model is over the whole answer |
| Answer log-prob (neg) | how unsure the model is on the final answer tokens |
| Mean entropy | average spread of the next-token distribution |
| Output length | number of generated tokens |
| 3-sample disagreement | how often the other answers to the same question disagree (needs extra generations) |
| Supervised probe | logistic regression on the mean answer hidden state, trained **with** labels, out-of-fold. An upper bound, not a fair competitor |

Saved to `features.csv`.

In [ ]:
def _mean(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(x.mean()) if len(x) else np.nan

def _max(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(x.max()) if len(x) else np.nan

rows = []
for g in valid_ids:
    r = records[g]
    a, L, tf = r["a"], r["L"], TOKF[g]
    row = dict(gid=g, qid=r["qid"], sample_idx=r["sample_idx"], fold=r["fold"], pred=r["pred"],
               gold=gold[r["qid"]], correct=r["correct"], y_wrong=int(not r["correct"]),
               changed=r["changed"], n_gen=r["n_gen"], a=a, L=L, truncated=r["truncated"],
               b_neg_mean_logprob=float(tf["surp"].mean()),
               b_neg_answer_logprob=float(tf["surp"][a:a + L].mean()),
               b_mean_entropy=float(tf["ent"].mean()),
               b_length=float(r["n_gen"]),
               b_disagreement=1.0 - r["agreement"])
    for d in DEPTHS:
        for an in ANCHORS:
            base = f"tscR_{an}_{DT[d]}"
            rho, adj = TSC[(d, an, "real")][g], ADJ_R[(d, an)].get(g)
            row[base + "_raw"], row[base + "_rawmax"] = _mean(rho), _max(rho)
            row[base + "_adj"] = _mean(adj) if adj is not None else np.nan
            row[base + "_adjmax"] = _max(adj) if adj is not None else np.nan
            row[base + "_rank"] = RANK[(d, an)][g]
            for null in ("random", "shuffled", "other"):
                arr = TSC[(d, an, null)].get(g)
                row[f"null_{null}_{an}_{DT[d]}"] = _mean(arr) if arr is not None else np.nan
        row[f"tscR_pre_{DT[d]}"] = PRE[d][g]
        loc = LOCAL[d][g][0][a:a + L]
        row[f"tscL_k{LOCAL_K}_{DT[d]}_raw"] = _mean(loc)
        adjl = ADJ_L[d].get(g)
        row[f"tscL_k{LOCAL_K}_{DT[d]}_adj"] = _mean(adjl) if adjl is not None else np.nan
    rows.append(row)
F = pd.DataFrame(rows)

# Supervised upper bound: linear probe on the mean answer-token state, out-of-fold by question
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Xp = np.stack([standardise(torch.tensor(ANS_RAW[PRIMARY_DEPTH][g], device=DEV), records[g]["fold"],
                           PRIMARY_DEPTH).mean(0).cpu().numpy() for g in F["gid"]])
yp, grp = F["y_wrong"].values, F["qid"].values
oof = np.full(len(F), np.nan)
n_splits = min(5, len(np.unique(grp)))
if n_splits >= 2 and 0 < yp.sum() < len(yp):
    for tr, te in GroupKFold(n_splits=n_splits).split(Xp, yp, grp):
        if len(np.unique(yp[tr])) < 2:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=3000))
        oof[te] = clf.fit(Xp[tr], yp[tr]).predict_proba(Xp[te])[:, 1]
F["sup_probe_oof"] = oof
F.to_csv(os.path.join(OUT_DIR, "features.csv"), index=False)
print("feature table:", F.shape)

## 13. Metrics and main comparison

- **AUROC**: how well each score separates wrong from correct answers. 0.5 = chance, 1.0 = perfect.
- **95% CI**: from 2,000 bootstrap resamples over **questions** (answers to the same question stay together).
- **AUPRC** and **TPR at 5% FPR**: two more views of the same separation.
- **Within-question AUROC**: compares wrong and correct answers to the *same* question only. This removes the effect of question difficulty.

The second table is an **exploratory sweep** over all depths and anchors. It is not used for the decision.

In [ ]:
from sklearn.metrics import average_precision_score, roc_curve

def auroc(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    rk = rankdata(s)
    return float((rk[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def tpr_at_fpr(y, s, target=0.05):
    m = np.isfinite(s)
    if len(np.unique(y[m])) < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y[m], s[m])
    return float(np.interp(target, fpr, tpr))

def auprc(y, s):
    m = np.isfinite(s)
    return float(average_precision_score(y[m], s[m])) if len(np.unique(y[m])) == 2 else np.nan

def make_boot(qids, B=N_BOOT, seed=SEED + 99):
    qids = np.asarray(qids)
    groups = {q: np.where(qids == q)[0] for q in np.unique(qids)}
    keys = np.array(list(groups))
    rng = np.random.default_rng(seed)
    return [np.concatenate([groups[q] for q in rng.choice(keys, len(keys), replace=True)]) for _ in range(B)]

def ci(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if len(vals) > 10 else (np.nan, np.nan)

Yw = F["y_wrong"].values
BOOT = make_boot(F["qid"].values)

def auc_ci(col, y=Yw, frame=F, boot=BOOT):
    s = frame[col].values.astype(float)
    return auroc(y, s), ci([auroc(y[ix], s[ix]) for ix in boot])

def delta_ci(col_a, col_b, y=Yw, frame=F, boot=BOOT):
    sa, sb = frame[col_a].values.astype(float), frame[col_b].values.astype(float)
    ok = np.isfinite(sa) & np.isfinite(sb)
    point = auroc(y[ok], sa[ok]) - auroc(y[ok], sb[ok])
    bs = []
    for ix in boot:
        ix = ix[ok[ix]]
        bs.append(auroc(y[ix], sa[ix]) - auroc(y[ix], sb[ix]))
    return point, ci(bs)

def within_question_auc(col):
    """Difficulty-controlled: wrong vs correct generations of the SAME question (mixed questions only)."""
    s = F[col].values.astype(float)
    num = den = 0.0
    for q, ix in F.groupby("qid").indices.items():
        w, c = s[ix][Yw[ix] == 1], s[ix][Yw[ix] == 0]
        w, c = w[np.isfinite(w)], c[np.isfinite(c)]
        if len(w) and len(c):
            num += (w[:, None] > c[None]).sum() + 0.5 * (w[:, None] == c[None]).sum()
            den += len(w) * len(c)
    return (num / den if den else np.nan), int(den)

PD, PA = DT[PRIMARY_DEPTH], PRIMARY_ANCHOR
PRIM_RAW, PRIM_ADJ = f"tscR_{PA}_{PD}_raw", f"tscR_{PA}_{PD}_adj"
LOGP = "b_neg_mean_logprob"
MAIN = [
    ("Mean token log-prob (neg)", LOGP),
    ("Answer log-prob (neg)", "b_neg_answer_logprob"),
    ("Mean entropy", "b_mean_entropy"),
    ("Output length", "b_length"),
    (f"{N_SAMPLES + 1}-sample disagreement*", "b_disagreement"),
    (f"TSC-Local k={LOCAL_K} raw", f"tscL_k{LOCAL_K}_{PD}_raw"),
    (f"TSC-Local k={LOCAL_K} adj", f"tscL_k{LOCAL_K}_{PD}_adj"),
    ("TSC-Reasoning raw", PRIM_RAW),
    ("TSC-Reasoning adj  [PRIMARY]", PRIM_ADJ),
    ("TSC-R pre-answer token", f"tscR_pre_{PD}"),
    ("Null 1: shuffled anchor", f"null_shuffled_{PA}_{PD}"),
    ("Null 2: random subspace", f"null_random_{PA}_{PD}"),
    ("Null 3: other-question anchor", f"null_other_{PA}_{PD}"),
    ("Supervised probe (upper bound)", "sup_probe_oof"),
]
tab = []
for name, col in MAIN:
    s = F[col].values.astype(float)
    a_, (lo, hi) = auc_ci(col)
    wq, npairs = within_question_auc(col)
    tab.append(dict(detector=name, AUROC=a_, CI_lo=lo, CI_hi=hi, AUPRC=auprc(Yw, s),
                    TPR_at_5FPR=tpr_at_fpr(Yw, s), within_q_AUROC=wq))
TAB = pd.DataFrame(tab)
print(f"\nMain comparison — depth {PRIMARY_DEPTH} (layer {LAYER_OF[PRIMARY_DEPTH]}), anchor {PA}; "
      f"n={len(F)} ({int(Yw.sum())} wrong). Higher score = predicted wrong.")
print(TAB.round(3).to_string(index=False))
print("* only 3 samples per question: under-powered relative to the usual 10")
print(f"within_q_AUROC uses {within_question_auc(LOGP)[1]} wrong-vs-correct pairs from the same question")

# Exploratory sweep (not used for the decision): every anchor x depth x raw/adj x mean/max
sweep = []
for d in DEPTHS:
    for an in ANCHORS:
        base = f"tscR_{an}_{DT[d]}"
        sweep.append(dict(depth=d, layer=LAYER_OF[d], anchor=an,
                          raw=auroc(Yw, F[base + "_raw"].values), raw_max=auroc(Yw, F[base + "_rawmax"].values),
                          adj=auroc(Yw, F[base + "_adj"].values), adj_max=auroc(Yw, F[base + "_adjmax"].values),
                          median_rank=float(F[base + "_rank"].median())))
SWEEP = pd.DataFrame(sweep)
print("\nExploratory sweep (TSC-Reasoning AUROC):")
print(SWEEP.round(3).to_string(index=False))

## 14. Hypothesis checks

| Check | Question |
|---|---|
| **H1** | Does adjusted TSC beat mean log-prob? (difference in AUROC with CI) |
| **H2** | How much of any raw gain survives the confound adjustment? Also: AUROC within length tertiles, and within the same question |
| **H3** | Does TSC with the real anchor beat each of the three nulls? |
| **H4** | Does TSC just react to "the sampled answer differs from the greedy one", rather than to wrongness? |

In [ ]:
res = {}
d_adj, ci_adj = delta_ci(PRIM_ADJ, LOGP)
d_raw, ci_raw = delta_ci(PRIM_RAW, LOGP)
res["H1_gain_adj_vs_logprob"] = (d_adj, ci_adj)
res["H1_gain_raw_vs_logprob"] = (d_raw, ci_raw)
print(f"\nH1  ΔAUROC adj TSC-R − logprob = {d_adj:+.3f} [{ci_adj[0]:+.3f}, {ci_adj[1]:+.3f}]")
print(f"    ΔAUROC raw TSC-R − logprob = {d_raw:+.3f} [{ci_raw[0]:+.3f}, {ci_raw[1]:+.3f}]")

retained = d_adj / d_raw if d_raw > 0 else np.nan
res["H2_retained_fraction"] = retained
print(f"H2  adjusted gain keeps {retained:.0%} of the raw gain (falsified if < 50%)" if np.isfinite(retained)
      else "H2  raw gain <= 0, retention not defined")
F["len_tertile"] = pd.qcut(F["n_gen"].rank(method="first"), 3, labels=["short", "mid", "long"])
lt = []
for t_, sub in F.groupby("len_tertile", observed=True):
    y_ = sub["y_wrong"].values
    lt.append(dict(tertile=t_, n=len(sub), wrong=int(y_.sum()), tsc_adj=auroc(y_, sub[PRIM_ADJ].values),
                   logprob=auroc(y_, sub[LOGP].values), length=auroc(y_, sub["b_length"].values)))
print("    AUROC within length tertiles:")
print(pd.DataFrame(lt).round(3).to_string(index=False))
wq_t, _ = within_question_auc(PRIM_ADJ)
wq_l, _ = within_question_auc(LOGP)
print(f"    difficulty-controlled (same question) AUROC: TSC adj {wq_t:.3f} vs logprob {wq_l:.3f}")

h3 = {}
for null in ("shuffled", "random", "other"):
    dn, cn = delta_ci(PRIM_RAW, f"null_{null}_{PA}_{PD}")
    h3[null] = (dn, cn)
    print(f"H3  real − {null:8s} null: ΔAUROC = {dn:+.3f} [{cn[0]:+.3f}, {cn[1]:+.3f}]")
res["H3"] = h3

S = F[(F["sample_idx"] > 0) & F["changed"].notna()].copy()
S["changed"] = S["changed"].astype(bool)
def group_auc(sub_pos, sub_neg, col):
    if len(sub_pos) < 5 or len(sub_neg) < 5:
        return np.nan, (np.nan, np.nan), len(sub_pos), len(sub_neg)
    frame = pd.concat([sub_pos.assign(_y=1), sub_neg.assign(_y=0)], ignore_index=True)
    y_, s_ = frame["_y"].values, frame[col].values.astype(float)
    boot = make_boot(frame["qid"].values, B=min(N_BOOT, 1000))
    return auroc(y_, s_), ci([auroc(y_[ix], s_[ix]) for ix in boot]), len(sub_pos), len(sub_neg)

cc, ck = S[S["correct"] & S["changed"]], S[S["correct"] & ~S["changed"]]
wc, wk = S[~S["correct"] & S["changed"]], S[~S["correct"] & ~S["changed"]]
h4 = {}
for label, pos, neg in [("correct-changed vs correct-kept", cc, ck),
                        ("changed: wrong vs correct", wc, cc),
                        ("kept: wrong vs correct", wk, ck)]:
    for nm, col in (("TSC adj", PRIM_ADJ), ("logprob", LOGP)):
        a_, c_, n1, n0 = group_auc(pos, neg, col)
        h4[f"{label} | {nm}"] = (a_, c_, n1, n0)
        print(f"H4  {label:33s} {nm:8s} AUROC={a_:.3f} [{c_[0]:.3f}, {c_[1]:.3f}]  (n={n1} vs {n0})")
res["H4"] = h4

## 15. Go / no-go decision

The rule was set before the run. **GO** only if all four pass:

| | Criterion |
|---|---|
| G1 | adjusted TSC beats mean log-prob by ≥ 0.03 AUROC |
| G2a | TSC beats the shuffled-anchor null (CI above 0) |
| G2b | TSC beats the other-question null (CI above 0) |
| G3 | TSC does not just detect "answer changed" (AUROC ≤ 0.60) |

In [ ]:
auc_change = h4["correct-changed vs correct-kept | TSC adj"][0]
crit = {
    "G1: adj TSC-R beats mean log-prob by >= 0.03 AUROC": bool(np.isfinite(d_adj) and d_adj >= 0.03),
    "G2a: beats shuffled-anchor null (CI > 0)": bool(np.isfinite(h3["shuffled"][1][0]) and h3["shuffled"][1][0] > 0),
    "G2b: beats other-question null (CI > 0)": bool(np.isfinite(h3["other"][1][0]) and h3["other"][1][0] > 0),
    "G3: correct-changed vs correct-kept AUROC <= 0.60": bool(np.isfinite(auc_change) and auc_change <= 0.60),
}
print("\n================ GO / NO-GO ================")
for k_, v_ in crit.items():
    print(("PASS  " if v_ else "FAIL  ") + k_)
decision = "GO" if all(crit.values()) else "NO-GO"
print("Decision:", decision)
print("Note: 250 questions gives wide CIs; treat a borderline result as 'scale up the pilot', not as a verdict.")

## 16. Figure and saved results

Draws the summary figure and saves everything to `OUT_DIR`:

| File | Content |
|---|---|
| `results.json` | main table, sweep, hypothesis checks, go/no-go decision |
| `main_table.csv` | main comparison table |
| `sweep.csv` | exploratory depth × anchor sweep |
| `features.csv` | one row per valid answer with all scores |
| `tsc_pilot_results.png` | summary figure |

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
T2 = TAB.dropna(subset=["AUROC"]).iloc[::-1]
yerr = np.nan_to_num(np.vstack([T2["AUROC"] - T2["CI_lo"], T2["CI_hi"] - T2["AUROC"]]).clip(min=0))
colors = ["#c0392b" if "PRIMARY" in n else ("#7f8c8d" if n.startswith("Null") else "#2c7fb8") for n in T2["detector"]]
axes[0].barh(T2["detector"], T2["AUROC"] - 0.5, left=0.5, xerr=yerr, color=colors, capsize=3)
axes[0].axvline(0.5, color="k", lw=0.8)
axes[0].set_xlabel("AUROC (wrong vs correct), 95% cluster-bootstrap CI")
axes[0].set_title(f"{MODEL_NAME.split('/')[-1]} · MATH500[:{N_QUESTIONS}] · n={len(F)}")
for lab, colr, msk in (("correct", "#2c7fb8", F["y_wrong"] == 0), ("wrong", "#c0392b", F["y_wrong"] == 1)):
    axes[1].hist(F.loc[msk, PRIM_ADJ].dropna(), bins=30, alpha=0.6, label=lab, color=colr, density=True)
axes[1].set_xlabel("adjusted TSC-Reasoning score (excess residual)")
axes[1].set_title(f"Primary score by correctness (layer {LAYER_OF[PRIMARY_DEPTH]}, {PA})")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tsc_pilot_results.png"), dpi=150)
plt.show()

def _jsonable(o):
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    return o

summary = dict(model=MODEL_NAME, dtype=str(DTYPE), n_questions=len(X), n_generations=len(records),
               n_valid=len(F), n_wrong=int(Yw.sum()), primary=dict(depth=PRIMARY_DEPTH, layer=LAYER_OF[PRIMARY_DEPTH],
               anchor=PRIMARY_ANCHOR, tau=TAU), main_table=TAB.to_dict("records"), sweep=SWEEP.to_dict("records"),
               hypotheses=res, go_criteria=crit, decision=decision, math_verify=HAVE_MV)
with open(os.path.join(OUT_DIR, "results.json"), "w") as fh:
    json.dump(_jsonable(summary), fh, indent=2)
TAB.to_csv(os.path.join(OUT_DIR, "main_table.csv"), index=False)
SWEEP.to_csv(os.path.join(OUT_DIR, "sweep.csv"), index=False)
print("saved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

## 17. Results of the pilot run

These are the results from the run on Kaggle (2 × T4). The output files are in the `results/` folder of this repo.

**Answers analysed:** 681 of 750 were valid (430 correct, 251 wrong).

![Results](results/tsc_pilot_results.png)

| Detector | AUROC | 95% CI |
|---|---|---|
| Mean token log-prob (neg) | 0.705 | 0.655 – 0.753 |
| Answer log-prob (neg) | 0.815 | 0.772 – 0.853 |
| Mean entropy | 0.729 | 0.679 – 0.777 |
| Output length | 0.727 | 0.678 – 0.773 |
| 3-sample disagreement | 0.926 | 0.899 – 0.950 |
| TSC-Reasoning, raw | 0.496 | 0.437 – 0.550 |
| **TSC-Reasoning, adjusted (primary)** | 0.452 | 0.395 – 0.507 |
| Null 1: shuffled anchor | 0.610 | 0.561 – 0.656 |
| Null 2: random subspace | 0.480 | 0.436 – 0.526 |
| Null 3: other-question anchor | 0.552 | 0.500 – 0.603 |
| Supervised probe (uses labels) | 0.832 | 0.788 – 0.868 |

**Go / no-go:** G1 ❌ · G2a ❌ · G2b ❌ · G3 ✅ → **NO-GO**

### Summary

- The primary TSC score did **not** detect wrong answers. Its AUROC was 0.452, and the CI includes chance (0.5).
- Simple baselines did much better: answer log-prob reached 0.815, and 3-sample disagreement reached 0.926.
- The hidden states do carry the signal, since a supervised probe reached 0.832. But this unsupervised geometric score does not pick it up.
- TSC did not beat its null versions. The shuffled-anchor null even scored higher (0.610).

These results are for one small model on 250 questions. They may differ for other models or tasks.